# Small U-Net for STL-10

A small U-Net for STL-10 (96x96x3 RGB natural images), built from
plain `torch.nn` layers only -- no external U-Net/model library. This
is the first skip-connection architecture in the repo, meant to be
compared directly against the plain convolutional autoencoder
(`code/stl10_conv_autoencoder.py`) on the same task and loss:

```
L(x) = mean((x - unet(x))^2)        # pixel-wise MSE, same loss as the autoencoder
```

**Key difference from the autoencoder**: the autoencoder squeezes the
whole image through a single FC bottleneck vector `z in R^128`, so any
detail not captured by those 128 numbers is permanently lost. The
U-Net instead keeps one skip connection per resolution level -- the
*full* encoder feature map at each spatial size (96x96, 48x48, 24x24)
is concatenated (channel-wise) with the matching decoder feature map,
so fine spatial detail can bypass the bottleneck entirely. The
12x12 bottleneck here is a convolutional feature map, not a flattened
vector -- there is no FC layer anywhere in this model.

**Architecture** (`SmallUNet`, defined in `code/stl10_unet.py`), shape
trace for input `[B, 3, 96, 96]`:

```
--- encoder ---
enc0_conv (3  ->32 , k3,s1,p1) -> [B, 32, 96, 96]         (-> skip0)
down0     (32 ->64 , k3,s2,p1) -> [B, 64, 48, 48]
enc1_conv (64 ->64 , k3,s1,p1) -> [B, 64, 48, 48]         (-> skip1)
down1     (64 ->128, k3,s2,p1) -> [B,128, 24, 24]
enc2_conv (128->128, k3,s1,p1) -> [B,128, 24, 24]         (-> skip2)
down2     (128->256, k3,s2,p1) -> [B,256, 12, 12]
--- bottleneck ---
bottleneck_conv (256->256, k3,s1,p1) -> [B,256, 12, 12]   (no FC layer)
--- decoder ---
up2       (256->128, k3,s2,p1,op1) -> [B,128, 24, 24]
cat([up2, skip2], dim=1)           -> [B,256, 24, 24]     (128+128 channels)
dec2_conv (256->128, k3,s1,p1)     -> [B,128, 24, 24]
up1       (128->64 , k3,s2,p1,op1) -> [B, 64, 48, 48]
cat([up1, skip1], dim=1)           -> [B,128, 48, 48]     (64+64 channels)
dec1_conv (128->64 , k3,s1,p1)     -> [B, 64, 48, 48]
up0       (64 ->32 , k3,s2,p1,op1) -> [B, 32, 96, 96]
cat([up0, skip0], dim=1)           -> [B, 64, 96, 96]     (32+32 channels)
dec0_conv (64 ->32 , k3,s1,p1)     -> [B, 32, 96, 96]
out_conv  (32 ->3  , k1,s1,p0)     -> [B,  3, 96, 96]
sigmoid                            -> [B,  3, 96, 96]     (x_hat in [0,1])
```

Stride-2 3x3 convolutions (padding=1) halve `H,W` exactly at each
encoder stage (`out = floor((in+2*1-3)/2)+1`); 96 is divisible by 8
(2^3 downsampling stages), so every halving (96->48->24->12) lands on
an even number. Each decoder `ConvTranspose2d` uses `output_padding=1`
to recover the exact 2x upsampling
(`out = (in-1)*stride - 2*pad + kernel + output_padding`), which is
required for `torch.cat` to succeed: concatenation along the channel
dimension needs identical `H,W` on both tensors.

**Note on scale**: STL-10 is a ~2.6GB download, so this notebook *must*
run on Colab with GPU + Drive caching -- do not attempt real training
locally.

Core code lives in `code/stl10_unet.py` (version-controlled); this
notebook is a thin orchestration layer: GPU check, repo pull, build
model, print the full shape trace (every skip-save and every
before/after concatenation), download real STL-10 (Drive-cached), train,
plot the loss curve, and show original-vs-reconstruction qualitative +
quantitative results.

## 1. GPU check

In [ ]:
import torch

# Never hard-code CUDA: fall back to CPU if no GPU is available.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device = {device}")
if device.type == "cuda":
    print(torch.cuda.get_device_name(0))

## 2. Clone/pull repository

Model code lives in `code/stl10_unet.py`, not duplicated in this
notebook. Replace `REPO_URL` with this repository's URL.

In [ ]:
import os

REPO_URL = "https://github.com/RoeiZ/from_zero_to_hero.git"
REPO_DIR = "/content/from_zero_to_hero"

if not os.path.isdir(REPO_DIR):
    !git clone $REPO_URL $REPO_DIR
else:
    !cd $REPO_DIR && git pull

import sys
sys.path.insert(0, os.path.join(REPO_DIR, "code"))

## 3. Build model, print full shape trace (skip connections + concatenations)

One verbose forward pass on a small synthetic batch, purely to print
the `[B,C,H,W]` shape after every encoder/decoder op -- including the
shape of each skip tensor at the moment it is saved, and the shapes of
both tensors immediately before, and the fused tensor immediately
after, every `torch.cat`. Not part of training.

In [ ]:
from stl10_unet import SmallUNet, count_trainable_parameters

torch.manual_seed(0)

model = SmallUNet().to(device)
print(f"trainable parameters: {count_trainable_parameters(model):,}")

sample_batch = torch.rand(4, 3, 96, 96, device=device)  # [B,3,96,96], synthetic, in [0,1]
_ = model(sample_batch, verbose=True)

## 4. Download real STL-10 (train/test split), cached in Drive

Same dataset loader and train/test convention as
`code/stl10_conv_autoencoder.py`: `torchvision.datasets.STL10(split="train")`
/ `split="test")` gives the dataset's native labeled train/test
partition (the separate 100k-image `unlabeled` split is not used).

Colab's local disk (`/content/...`) is wiped whenever the runtime
recycles, so this ~2.6GB dataset would otherwise be re-downloaded on
every fresh session. Instead we mount Google Drive and point
`DATA_ROOT` at a persistent Drive folder: `torchvision` only downloads
into it once and simply reads from it on every later run.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

# Persistent cache: downloaded once into Drive, reused on every
# future Colab session (local /content storage is ephemeral).
DATA_ROOT = "/content/drive/MyDrive/thesis_data/stl10"

In [ ]:
from stl10_unet import get_stl10_dataloaders

BATCH_SIZE = 64  # 96x96x3 images use much more GPU memory per sample than MNIST/CIFAR-10

train_loader, test_loader = get_stl10_dataloaders(DATA_ROOT, BATCH_SIZE)
print(f"train batches = {len(train_loader)}, test batches = {len(test_loader)}")

## 5. Train

Explicit train/eval loops (`train_one_epoch`, `evaluate`), MSE
reconstruction loss (image vs. its own reconstruction, no labels used),
Adam optimizer -- identical training setup to the autoencoder notebook,
so the comparison isolates the architecture, not the optimizer/loss.

In [ ]:
import torch.nn as nn
from stl10_unet import train_one_epoch, evaluate

N_EPOCHS = 15
LR = 1e-3

loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

history = {"train_mse": [], "test_mse": []}

for epoch in range(N_EPOCHS):
    train_mse = train_one_epoch(model, train_loader, loss_fn, optimizer, device)
    test_mse = evaluate(model, test_loader, loss_fn, device)

    history["train_mse"].append(train_mse)
    history["test_mse"].append(test_mse)

    print(f"epoch {epoch + 1:3d}/{N_EPOCHS}  train_mse={train_mse:.4f}  test_mse={test_mse:.4f}")

## 6. Reconstruction-loss curve (quantitative result over training)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(history["train_mse"], label="train")
ax.plot(history["test_mse"], label="test")
ax.set_xlabel("epoch")
ax.set_ylabel("pixel-wise MSE")
ax.set_title("Reconstruction loss (U-Net)")
ax.legend()
fig.tight_layout()
plt.show()

print(f"final test_mse = {history['test_mse'][-1]:.4f}")

## 7. Qualitative + quantitative check: original vs. reconstruction

Per-project convention: always look at concrete examples, not only
scalar metrics. Show 5 real test images (top row) next to the model's
reconstruction (bottom row), with per-image MSE, PSNR, and SSIM
(structural similarity, Wang et al. 2004) printed under each
reconstruction. Compare this figure qualitatively against
`stl10_autoencoder_reconstructions.png` from the autoencoder notebook
-- the U-Net's skip connections should preserve texture/edge detail
that the FC-bottleneck autoencoder blurs away.

In [ ]:
from stl10_unet import visualize_reconstructions

sample_images, _sample_labels = next(iter(test_loader))
sample_images = sample_images.to(device)

visualize_reconstructions(model, sample_images, num_examples=5)

## 8. Test-set average reconstruction MSE and SSIM

Aggregate numbers over the full test split (not just the 5 plotted
examples), for direct comparison against the autoencoder's
`final_test_mse` from its own notebook (same dataset, same train/test
split, same loss -- only the architecture differs).

In [ ]:
from stl10_unet import evaluate_ssim

final_test_mse = evaluate(model, test_loader, loss_fn, device)
final_test_ssim = evaluate_ssim(model, test_loader, device)
print(f"average test-set reconstruction MSE (U-Net):  {final_test_mse:.4f}")
print(f"average test-set SSIM (U-Net):                {final_test_ssim:.4f}")